In [ ]:
# Import Necessary Libraries
import os
from __future__ import annotations

import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

In [ ]:
# Define the project path
PROJECT_ROOT = (
    Path.cwd().parent
)  # gets the notebook’s current working directory and moves one level upward. Because the notebook is inside notebooks/, its parent should be the repository root

RAW_DATA_DIR = (
    PROJECT_ROOT / "data" / "raw_data"
)  # creates the path for unprocessed HTML and CSV files.
CLEANED_DATA_DIR = (
    PROJECT_ROOT / "data" / "cleaned_data"
)  # creates the path for transformed output.
SCREENSHOTS_DIR = (
    PROJECT_ROOT / "screenshots"
)  # creates the path for project evidence images.
ENV_PATH = PROJECT_ROOT / ".env"  # points to the environment-variable file.

# Each .mkdir(parents=True, exist_ok=True) creates the folder and any missing parent folders.
# parents=True allows Python to create intermediate folders.
# exist_ok=True prevents an error when the folder already exists.
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOTS_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)  # displays the project root so that you can verify the path.

In [ ]:
BASE_URL = "https://books.toscrape.com/"  # BASE_URL stores the root address of Books to Scrape.
BOOK_BASE_URL = urljoin(BASE_URL, "catalogue/")  # The trailing slash keeps catalogue in joined book URLs.
START_URL = "https://books.toscrape.com/catalogue/page-1.html"  # START_URL stores the first catalogue page, which is the starting point for pagination.

print(START_URL)  # confirms the selected starting page.

In [ ]:
# First HTTP request to the target URL
HEADERS = {  # creates a dictionary of request headers.
    "User-Agent": (  # identifies the scraper as PagePulseStudentBot/1.0.
        "PagePulseStudentBot/1.0 "
        "(educational web-scraping project)"  # This makes the client transparent
    )
}

session = (requests.Session())  # creates a persistent HTTP session that can reuse connections.
session.headers.update(HEADERS)  # applies the custom header to every request sent through the session.

In [ ]:
# First page request to the target URL
response = session.get(START_URL, timeout=30)  # sends an HTTP GET request

print("Status code:", response.status_code)  # shows the HTTP status
print(
    "Content type:", response.headers.get("Content-Type")
)  # shows the type of content returned.
print(
    "HTML length:", len(response.text)
)  # shows the number of decoded characters in the HTML response.

# Purpose: Downloads the first catalogue page and inspects the response.

In [ ]:
# Do not clean the HTML before saving it. Raw data should represent what the source returned.

# 'datetime' is a module in Python that handles dates and time
# 'now' means 'give me the current time this very second'
# 'timezone.utc' this is the world's standard clock/time. This of UTC has the master clock that everyone agrees on
scraped_at = datetime.now(
    timezone.utc
)  # This code is asking for the current universal time. It saves time in universal standard time.

raw_html_path = RAW_DATA_DIR / (
    f"catalogue_page_1_{scraped_at:%Y%m%d_%H%M%S}.html"
)  # builds the output file path.

raw_html_path.write_text(
    response.text,
    encoding="utf-8",  # This makes sure all characters are saved correctly
)  # writes the HTML to disk using UTF-8 encoding

print(
    "Raw HTML saved to:", raw_html_path
)  # The final print reports the saved location.

In [ ]:
# Parsed document
soup = BeautifulSoup(
    response.text, "lxml"  # lxml is the best parser/translator for HTML and XML files
)  # converts the raw HTML string into a searchable document tree

In [ ]:
# Inspect the page title
print(soup.title)  # prints the title of the page
print(soup.title.get_text(strip=True))  # prints the text of the title

In [ ]:
# Select the first book card
first_book = soup.select_one(
    "article.product_pod"
)  # This locates the first book card on the website

print(first_book)  # This will display the card's raw HTML for inspection

In [ ]:
# Extract the book title, price, availability, rating, product URL, Image URL from the first book card

# Each .select_one(...) searches inside the first book card.
title_element = first_book.select_one("h3 a")  # "h3 a" locates the title link.
price_element = first_book.select_one(
    "p.price_color"
)  # "p.price_color" locates the price.
availability_element = first_book.select_one(
    "p.instock.availability"
)  # "p.instock.availability" locates the stock text.
rating_element = first_book.select_one(
    "p.star-rating"
)  # "p.star-rating" locates the rating element.
image_element = first_book.select_one(
    "img.thumbnail"
)  # "img.thumbnail" locates the image.

title = title_element.get(
    "title"
)  # title_element.get("title") retrieves the full title attribute.
price_text = price_element.get_text(
    strip=True
)  # .get_text(strip=True) extracts and trims the price text.
availability_text = availability_element.get_text(
    " ", strip=True
)  # .get_text(" ", strip=True) joins availability text with spaces and removes outer whitespace.

rating_classes = rating_element.get(
    "class", []
)  # retrieves the rating element’s CSS classes.
rating_text = next(
    (
        value for value in rating_classes if value != "star-rating"
    ),  # returns the first class that is not star-rating, such as Three.
    None,  # None is returned if no rating class is found.
)

## converts relative product and image links into absolute URLs.
book_url = urljoin(BOOK_BASE_URL, title_element.get("href"))

image_url = urljoin(START_URL, image_element.get("src"))

# display every extracted field.
print("Title:", title)
print("Price:", price_text)
print("Availability:", availability_text)
print("Rating:", rating_text)
print("Book URL:", book_url)
print("Image URL:", image_url)

In [ ]:
# Define a reusable function that scrapes one catalogue page.


def scrape_catalogue_page(  # a function called scrape_catalogue_page is defined with the def keyword
    page_url: str,  # str means the function expects a page URL as text.
    session: requests.Session,  # means it expects the shared HTTP session.
) -> tuple[
    list[dict], str | None
]:  # means it returns a list of book dictionaries and either the next-page URL or None.
    """Scrape all books and the next-page URL from one catalogue page."""

    response = session.get(page_url, timeout=30)  # downloads the requested page.
    response.raise_for_status()  # stops the function on an HTTP error.

    soup = BeautifulSoup(response.text, "lxml")  # parses the HTML.

    records: list[dict] = []  # creates an empty result list.

    for book in soup.select("article.product_pod"):  # loops through every book card.
        title_element = book.select_one(
            "h3 a"
        )  # The five selector statements locate the relevant child elements.
        price_element = book.select_one("p.price_color")
        availability_element = book.select_one("p.instock.availability")
        rating_element = book.select_one("p.star-rating")
        image_element = book.select_one("img.thumbnail")

        rating_classes = (
            rating_element.get("class", []) if rating_element else []
        )  # The conditional rating_classes expression avoids calling .get() on a missing element.

        rating_text = next(
            (
                value for value in rating_classes if value != "star-rating"
            ),  # extracts the word-based rating class.
            None,  # None is returned if it is not found.
        )

        # 'record'creates one dictionary for the current book.
        # Each conditional expression returns the extracted value when the element exists and None otherwise.
        record = {
            "title": (title_element.get("title") if title_element else None),
            "price_raw": (
                price_element.get_text(strip=True) if price_element else None
            ),
            "availability_raw": (
                availability_element.get_text(" ", strip=True)
                if availability_element
                else None
            ),
            "rating_raw": rating_text,
            "book_url": (
                urljoin(BOOK_BASE_URL, title_element["href"])
                if title_element and title_element.get("href")
                else None
            ),
            "image_url": (
                urljoin(page_url, image_element["src"])
                if image_element and image_element.get("src")
                else None
            ),
            "source_page": page_url,  # page_url records where the row came from.
            "scraped_at": datetime.now(timezone.utc),  # records extraction time.
        }

        records.append(record)  # adds the book to the page result.

    next_element = soup.select_one("li.next a")  # locates the pagination/page link.

    next_url = (  # becomes the absolute next-page address or None on the final page.
        urljoin(
            page_url, next_element.get("href")
        )  # creates complete product and image URLs.
        if next_element and next_element.get("href")
        else None
    )

    return records, next_url  # returns both outputs.

In [ ]:
# Test the function created above
page_one_records, next_page_url = (
    scrape_catalogue_page(  # Calls scrape_catalogue_page() on the first page.
        START_URL, session
    )
)  # Tuple unpacking, stores the book list in page_one_records and the next page in next_page_url

# The print statements confirm the number of extracted books and the next URL.
print("Books extracted:", len(page_one_records))
print("Next page:", next_page_url)

In [ ]:
# Limit the run to 3 pages while testing
all_books: list[dict] = []  # creates the combined result list.

current_url = START_URL  # sets the first page
maximum_pages = 10  # limits the test
page_number = 1  # initializes the counter

while (
    current_url and page_number <= maximum_pages
):  # The while condition requires both a valid URL and a page number within the test limit.
    print(
        f"Scraping page {page_number}: {current_url}"
    )  # The progress print shows the current page.

    page_records, next_url = scrape_catalogue_page(
        current_url, session
    )  # This scrapes one page

    all_books.extend(page_records)  # adds every book dictionary to the combined list.

    current_url = next_url
    page_number += 1  # increases the counter

    time.sleep(1)  # Pauses for one second between requests

print("Total records:", len(all_books))  # This shows the combined number of records

In [ ]:
all_books = []
current_url = START_URL
page_number = 1  # Resets the result list, URL and page counter.

while current_url:  # continues until the function returns None
    print(f"Scraping page {page_number}")  # This shows the page number

    page_records, current_url = scrape_catalogue_page(
        current_url, session
    )  # stores the current page’s books and immediately replaces current_url with the next-page URL.

    all_books.extend(page_records)  # This adds all page records

    page_number += 1  # The counter is incremented
    time.sleep(1)  # adds a polite delay of one second
    # The loop stops after the last page

print(
    "Total records scraped:", len(all_books)
)  # This prints a report of the total number of books scraped

In [ ]:
books_raw = pd.DataFrame(all_books)  # converts the list of dictionaries into a DataFrame

In [ ]:
books_raw.head()  # displays the first five rows of the DataFrame

In [ ]:
books_raw.columns  # This shows the column names in the DataFrame

In [ ]:
raw_csv_path = RAW_DATA_DIR / (
    f"books_raw_{scraped_at:%Y%m%d_%H%M%S}.csv"
)  # This builds a timestamped path inside data/raw_data directory

books_raw.to_csv(
    raw_csv_path,
    index=False,  # prevents the Pandas row index from becoming a CSV column.
)  # Writes the DataFrame to CSV

print("Raw CSV saved to:", raw_csv_path)  # This confirms the saved path